# Data Generation

This notebook establishes the foundation of the project by generating a synthetic e-commerce dataset that reflects realistic user behavior, device usage, cart values, and checkout outcomes.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

NUM_RECORDS = 5000
user_pool = [f"USER_{i:04d}" for i in range(1, 1501)]
user_ids = np.random.choice(user_pool, size=NUM_RECORDS)

start_date = pd.Timestamp('2026-01-01')
end_date = pd.Timestamp('2026-06-30')
total_seconds = int((end_date - start_date).total_seconds())
random_seconds = np.random.randint(0, total_seconds, size=NUM_RECORDS)
session_start_times = [start_date + pd.Timedelta(seconds=int(sec)) for sec in random_seconds]

df = pd.DataFrame({
    'user_id': user_ids,
    'session_start_time': session_start_times
}).sort_values(by='session_start_time').reset_index(drop=True)

df['device_type'] = np.random.choice(['Mobile', 'Desktop', 'Tablet'], size=NUM_RECORDS, p=[0.55, 0.35, 0.10])
raw_cart_values = np.random.lognormal(mean=4.0, sigma=0.65, size=NUM_RECORDS)
df['cart_value'] = np.round(np.clip(raw_cart_values, 10.0, 500.0), 2)

def calculate_abandonment_prob(row):
    prob = 0.45
    if row['device_type'] == 'Mobile':
        prob += 0.20
    elif row['device_type'] == 'Desktop':
        prob -= 0.10

    hour = row['session_start_time'].hour
    if 0 <= hour <= 5:
        prob += 0.12
    elif 18 <= hour <= 22:
        prob -= 0.05

    if row['cart_value'] > 200:
        prob += 0.10

    return np.clip(prob, 0.15, 0.85)

abandon_probs = df.apply(calculate_abandonment_prob, axis=1)
random_draws = np.random.rand(NUM_RECORDS)
df['checkout_status'] = np.where(random_draws < abandon_probs, 'Abandoned', 'Completed')

cart_val_missing_idx = np.random.choice(NUM_RECORDS, size=int(NUM_RECORDS * 0.04), replace=False)
device_missing_idx = np.random.choice(NUM_RECORDS, size=int(NUM_RECORDS * 0.04), replace=False)
df.loc[cart_val_missing_idx, 'cart_value'] = np.nan
df.loc[device_missing_idx, 'device_type'] = np.nan

df.to_csv('ecommerce_cart_data_raw.csv', index=False)
print('Synthetic dataset saved to ecommerce_cart_data_raw.csv')
df.head()


e:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
e:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Synthetic dataset saved to ecommerce_cart_data_raw.csv


,user_id,session_start_time,device_type,cart_value,checkout_status
0,USER_0563,2026-01-01 00:15:17,Desktop,154.53,Abandoned
1,USER_0598,2026-01-01 01:06:51,Mobile,42.24,Abandoned
2,USER_0400,2026-01-01 01:47:44,Mobile,38.67,Abandoned
3,USER_0534,2026-01-01 03:34:39,NaN,24.78,Abandoned
4,USER_0476,2026-01-01 05:19:16,Tablet,90.41,Completed
